# Phase 5 — NB2: Neighbor Perturbation flip=0.4

**Goal:** Train Stage 2 with aggressive neighbor perturbation (flip_prob=0.4).
- Each neighbor's polarity independently flipped with 40% probability during training
- Matches neutral's 44% wrong-neighbor rate at test time
- Risk: too much noise may cause model to ignore retrieval entirely
- Config: `stage2_2014_perturb04.yaml`

**Input:**
- `duclm318/semeval-2014-absa-restaurant` — SemEval 2014 data
- `duclm318/p5-embed-v4` — MAMS embedding

**Output:** `outputs_p5_nb2/` — perturb04 checkpoint + training log

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml

In [ ]:
import os, sys, json, shutil

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

In [ ]:
# --- Wire SemEval 2014 XMLs ---
KAGGLE_INPUT = None
for candidate in ['/kaggle/input/semeval-2014-absa-restaurant',
                  '/kaggle/input/datasets/duclm318/semeval-2014-absa-restaurant',
                  '/kaggle/input/datasets/lcminhc/semeval-2014-absa-restaurant']:
    if os.path.exists(candidate):
        KAGGLE_INPUT = candidate
        break
assert KAGGLE_INPUT, 'Dataset semeval-2014-absa-restaurant not found'
print(f'XML Input: {KAGGLE_INPUT}')

os.makedirs('SemEval-2014', exist_ok=True)
shutil.copy(f'{KAGGLE_INPUT}/Restaurants_Train.xml',
            'SemEval-2014/Restaurants_Train.xml')
shutil.copy(f'{KAGGLE_INPUT}/Restaurants_Test_Gold.xml',
            'SemEval-2014/Restaurants_Test_Gold.xml')
print('SemEval 2014 XML files wired.')

# --- Prepare data (generates sentiment_records.jsonl) ---
!python scripts/01_prepare_data.py

# --- Wire NB0 embedding checkpoint ---
EMB = None
for candidate in ['/kaggle/input/p5-embed-v4',
                  '/kaggle/input/datasets/duclm318/p5-embed-v4',
                  '/kaggle/input/datasets/lcminhc/p5-embed-v4']:
    if os.path.exists(candidate):
        EMB = candidate
        break
assert EMB, 'Dataset p5-embed-v4 not found'
print(f'\nNB0 Input: {EMB} -> {os.listdir(EMB)}')

os.makedirs('checkpoints/embedding_2014', exist_ok=True)
shutil.copy(f'{EMB}/embedding_v4_s2_best.pt', 'checkpoints/embedding_2014/best.pt')
print(f'Embedding ckpt: {os.path.getsize("checkpoints/embedding_2014/best.pt") / 1e6:.1f} MB')

## 0b. Build FAISS Index

In [ ]:
os.makedirs('indexes', exist_ok=True)
!python scripts/03_build_index.py \
    --embedding_ckpt checkpoints/embedding_2014/best.pt \
    --input data/processed/sentiment_records.jsonl \
    --out_dir indexes/

for f in ['train.faiss', 'train_metadata.jsonl', 'train_vectors.npy']:
    path = f'indexes/{f}'
    if os.path.exists(path):
        print(f'{f}: {os.path.getsize(path)/1e6:.2f} MB')
    else:
        print(f'MISSING: {f}')

In [ ]:
import torch, gc
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Train — Neighbor Perturbation flip=0.4

Config: `stage2_2014_perturb04.yaml`
- `neighbor_flip_prob: 0.4` — each neighbor polarity independently flipped with 40% probability
- With k=3: ~78% of samples have at least 1 wrong neighbor
- Matches neutral's 44% wrong-neighbor rate observed at test time
- Risk: too much noise → model may ignore retrieval entirely

In [ ]:
gc.collect()
torch.cuda.empty_cache()
!python scripts/04b_train_stage2.py \
    --config configs/stage2_2014_perturb04.yaml \
    --embedding_ckpt checkpoints/embedding_2014/best.pt \
    --index_dir indexes/ \
    --retrieval_config configs/retrieval_v2.yaml

## 2. Training Log

In [ ]:
log_path = 'logs/stage2_2014_perturb04_training.jsonl'
print('=== Perturbation flip=0.4 ===')
if not os.path.exists(log_path):
    print('No log found.')
else:
    print(f'{"Epoch":<6} {"Loss":<8} {"Acc":<8} {"MacF1":<10} {"pos":<7} {"neg":<7} {"neu":<7}')
    print('-' * 55)
    with open(log_path) as f:
        for line in f:
            r = json.loads(line)
            print(f"{r['epoch']:<6} {r['train_loss']:<8.4f} "
                  f"{r['sentiment_acc']:<8.4f} {r['sentiment_macro_f1']:<10.4f}"
                  f"{r.get('f1_positive', 0):<7.3f} "
                  f"{r.get('f1_negative', 0):<7.3f} "
                  f"{r.get('f1_neutral', 0):<7.3f}")

## 3. Save Outputs

In [ ]:
output_dir = '/kaggle/working/outputs_p5_nb2'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f'{output_dir}/logs', exist_ok=True)

src = 'checkpoints/stage2_2014_perturb04/best.pt'
if os.path.exists(src):
    shutil.copy(src, f'{output_dir}/stage2_2014_perturb04_best.pt')
    print(f'stage2_2014_perturb04_best.pt: {os.path.getsize(src)/1e6:.1f} MB')

src = 'logs/stage2_2014_perturb04_training.jsonl'
if os.path.exists(src):
    shutil.copy(src, f'{output_dir}/logs/')

print(f'\nOutputs saved to {output_dir}')
print('Upload as Kaggle dataset: p5-nb2-stage2')

In [ ]:
shutil.make_archive('/kaggle/working/outputs_p5_nb2_backup', 'zip',
                    '/kaggle/working', 'outputs_p5_nb2')
size_mb = os.path.getsize('/kaggle/working/outputs_p5_nb2_backup.zip') / 1e6
print(f'Backup zip: {size_mb:.1f} MB')